In [6]:
import io
import time
from pathlib import Path

import pandas as pd
import requests

MAP_KEY = 'cbf821fc44c5cdb58ef790f8c1540286'
URL = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
SOURCE = "MODIS_SP"
# Your HydroViewer/Amazon domain
AREA = "-82.0,-21.0,-49.0, 6.0"

OUT_DIR = Path("./firms_modis")
OUT_DIR.mkdir(exist_ok=True)

try:
    response = requests.get(URL)
    data = response.json()
    df = pd.Series(data)
    print(df)
except:
  print ("There is an issue with the query. \nTry in your browser: %s" % URL)

transaction_limit             5000
current_transactions             2
transaction_interval    10 minutes
dtype: object


In [5]:
def get_available_dates():
    url = (
        "https://firms.modaps.eosdis.nasa.gov/"
        f"api/data_availability/csv/{MAP_KEY}/{SOURCE}"
    )

    df = pd.read_csv(url)

    return (
        pd.Timestamp(df.iloc[0]["min_date"]),
        pd.Timestamp(df.iloc[0]["max_date"]),
    )

start_date, end_date = get_available_dates()

print("Available:", start_date, "->", end_date)

Available: 2000-11-01 00:00:00 -> 2026-06-30 00:00:00


In [ ]:
def download_chunk(start_date, days=5):
    date_str = start_date.strftime("%Y-%m-%d")

    url = (
        f"{BASE_URL}/"
        f"{MAP_KEY}/"
        f"{SOURCE}/"
        f"{AREA}/"
        f"{days}/"
        f"{date_str}"
    )

    print(f"Downloading {date_str} ({days} days)")

    r = requests.get(url, timeout=120)
    r.raise_for_status()

    # FIRMS may legitimately return no detections
    if not r.text.strip():
        return pd.DataFrame()

    return pd.read_csv(io.StringIO(r.text))


current = start_date

while current <= end_date:

    remaining = (end_date - current).days + 1
    days = min(5, remaining)

    outfile = OUT_DIR / f"MODIS_SP_{current:%Y%m%d}.csv"

    # Makes script restartable
    if outfile.exists():
        print("Already exists:", outfile)
        current += pd.Timedelta(days=days)
        continue

    try:
        df = download_chunk(current, days)

        if not df.empty:
            df.to_csv(outfile, index=False)
            print(f"  -> {len(df):,} detections")
        else:
            print("  -> no detections")

    except Exception as e:
        print("ERROR:", current, e)

        # Wait and retry once
        time.sleep(30)

        try:
            df = download_chunk(current, days)

            if not df.empty:
                df.to_csv(outfile, index=False)

        except Exception as e:
            print("FAILED:", current, e)
            break

    # Don't hammer FIRMS
    time.sleep(1)

    current += pd.Timedelta(days=days)